# 3.34 — Survival Analysis & the Cox Model

Survival analysis models **time until an event** while respecting the fact that some examples are censored: we know they survived at least this long, but not when the event would eventually happen. The Cox proportional hazards model keeps time flexible through an unknown baseline hazard while learning multiplicative risk ratios from features.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build survival analysis one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math, including censoring, risk sets, Cox risk ratios, partial likelihood, and score-based model selection, is shown explicitly. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, sorting, products, exponentials, and small numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any simulated examples.

### 1. Time-to-event data and censoring

Survival data has three columns: a time, an event indicator, and covariates. If `event=1`, the event happened at that time. If `event=0`, the row is **right-censored**: the person was still event-free when observation stopped, so their true event time is later than the recorded time. The key rule is that censored rows are not failures, but they still contribute evidence up to the time they were last observed.

In [ ]:
time_w = np.array([2., 3., 4., 6., 7., 9.])  # observed event or censoring times.
event_w = np.array([1, 0, 1, 1, 0, 1])  # 1 = event, 0 = right-censored.
x_w = np.array([0., 1., 0., 1., 1., 0.])  # one binary risk feature.
print("time:", time_w)  # inspect the observed times.
print("event:", event_w, "  censored rows:", np.where(event_w == 0)[0])  # locate censoring.

▶ What you'll see: rows 1 and 4 are censored, so their event times are unknown beyond 3 and 7.

In [ ]:
order_w = np.argsort(time_w)  # sort rows in time order for survival calculations.
print("time order:", order_w)  # inspect chronological row order.
print("ordered (time,event,x):")  # print one row per subject.
for i_w in order_w:
    print(time_w[i_w], event_w[i_w], x_w[i_w])  # censored rows remain in the time line.

▶ What you'll see: censored observations are interleaved with events; they are not discarded.

In [ ]:
plt.figure(figsize=(5, 2.8))  # create a compact time-line figure.
colors_w = np.where(event_w == 1, "crimson", "gray")  # red for events, gray for censored.
plt.scatter(time_w, np.arange(len(time_w)), c=colors_w, s=80)  # plot one subject per row.
for i_w, t_w in enumerate(time_w):
    plt.hlines(i_w, 0, t_w, color="lightgray")  # show observed at-risk time.
plt.xlabel("observed time"); plt.ylabel("subject")  # label axes.
plt.title("1: event times and right-censoring"); plt.show()  # render the time line.

▶ What you'll see: red dots are events; gray dots are last-seen-alive censoring times.

*Why it's done this way:* ordinary regression would pretend every recorded time is a true event time, which turns censoring into fake failures. Survival analysis separates the observed duration from the event indicator so censored subjects influence risk sets until their censoring time, then stop contributing afterward.

### 2. Survival and hazard: probability of lasting versus instant risk

The survival function $S(t)=P(T>t)$ is the chance the event time exceeds $t$. The hazard $h(t)$ is an instantaneous event rate among those still at risk at time $t$. A small hazard decrement multiplies survival by roughly $e^{-h\Delta t}$, so survival curves fall as cumulative hazard grows.

In [ ]:
t_grid_w = np.arange(0, 8)  # simple discrete time grid.
hazard_w = np.array([0.00, 0.05, 0.05, 0.10, 0.10, 0.15, 0.15, 0.20])  # per-interval hazard.
cumhaz_w = np.cumsum(hazard_w)  # cumulative hazard H(t).
surv_w = np.exp(-cumhaz_w)  # S(t)=exp(-H(t)) for this piecewise constant demo.
print("cumulative hazard:", np.round(cumhaz_w, 3))  # inspect H(t).
print("survival:", np.round(surv_w, 3))  # inspect S(t).

▶ What you'll see: cumulative hazard increases while survival decreases multiplicatively.

In [ ]:
assert round(float(surv_w[3]), 3) == 0.819  # exp(-(0.05+0.05+0.10)) = exp(-0.20).
plt.figure(figsize=(5, 3))  # create a survival plot.
plt.step(t_grid_w, surv_w, where="post", color="teal", label="S(t)")  # draw survival as a step curve.
plt.plot(t_grid_w, hazard_w, "o--", color="crimson", label="hazard")  # overlay hazards.
plt.ylim(0, 1.05); plt.xlabel("time"); plt.legend()  # keep scales readable.
plt.title("2: hazard accumulates into survival"); plt.show()  # display the figure.

▶ What you'll see: the survival curve drops faster when the hazard points get larger.

*Why it's done this way:* hazard is local risk among current survivors, while survival is the accumulated consequence of all previous risk. The exponential link $S(t)=e^{-H(t)}$ appears because many small multiplicative survival probabilities combine into an exponential of the summed hazard.

### 3. Kaplan–Meier estimates censoring-aware survival

The Kaplan–Meier estimator updates survival only at event times. At each event time, it multiplies the current survival by $1-d_j/n_j$, where $d_j$ is the number of events at that time and $n_j$ is the number still at risk just before that time. Censored subjects reduce later risk sets, but do not create survival drops at their censoring time.

In [ ]:
event_times_w = np.unique(time_w[event_w == 1])  # times where failures occur.
km_vals_w = []  # store survival after each event time.
S_now_w = 1.0  # survival starts at 1 before any events.
for t_w in event_times_w:
    n_w = np.sum(time_w >= t_w)  # still at risk just before t.
    d_w = np.sum((time_w == t_w) & (event_w == 1))  # events at t.
    S_now_w *= (1 - d_w / n_w)  # Kaplan-Meier multiplicative update.
    km_vals_w.append(S_now_w)  # save survival after t.
    print("t=", t_w, "n=", int(n_w), "d=", int(d_w), "S=", round(S_now_w, 3))  # inspect update.

▶ What you'll see: survival drops at 2, 4, 6, and 9, but not at censored times 3 and 7.

In [ ]:
km_vals_w = np.array(km_vals_w)  # convert to array for plotting.
assert round(float(km_vals_w[1]), 3) == 0.625  # after t=2 and t=4: (5/6)*(3/4)=0.625.
plot_t_w = np.r_[0, event_times_w]  # include time zero.
plot_s_w = np.r_[1.0, km_vals_w]  # include initial survival.
plt.figure(figsize=(5, 3))  # create a compact KM plot.
plt.step(plot_t_w, plot_s_w, where="post", color="purple")  # draw step survival.
plt.scatter(time_w[event_w == 0], np.interp(time_w[event_w == 0], plot_t_w, plot_s_w), marker="x", color="black", label="censored")  # mark censoring.
plt.ylim(0, 1.05); plt.xlabel("time"); plt.ylabel("S(t)"); plt.legend()  # label plot.
plt.title("3: Kaplan-Meier survival estimate"); plt.show()  # display.

▶ What you'll see: censoring marks appear on flat parts of the curve rather than as downward jumps.

*Why it's done this way:* the risk set denominator $n_j$ changes over time, so each event is compared only against people who could have had the event at that moment. Multiplying conditional survival probabilities gives an estimate of surviving all event times up to $t$.

### 4. Cox proportional hazards: multiplicative risk ratios

The Cox model says $$h(t\mid x)=h_0(t)e^{\beta^\top x}.$$ The baseline hazard $h_0(t)$ is the shared time pattern, and $e^{\beta^\top x}$ is a multiplicative risk ratio. For a one-unit feature increase, the hazard ratio is $e^\beta$: it does not depend on time, which is the **proportional hazards** assumption.

In [ ]:
beta_w = 0.7  # one-feature Cox coefficient.
hr_w = float(np.exp(beta_w))  # hazard ratio for x=1 versus x=0.
base_h_w = np.array([0.04, 0.06, 0.08, 0.10])  # arbitrary baseline hazards over time.
h0_w = base_h_w  # x=0 hazard.
h1_w = base_h_w * hr_w  # x=1 hazard under proportional hazards.
print("hazard ratio exp(beta):", round(hr_w, 3))  # inspect multiplicative effect.
print("x=0 hazard:", np.round(h0_w, 3))  # baseline.
print("x=1 hazard:", np.round(h1_w, 3))  # scaled baseline.

▶ What you'll see: every x=1 hazard is about 2.014 times the corresponding x=0 hazard.

In [ ]:
assert round(hr_w, 3) == 2.014  # exp(0.7).
plt.figure(figsize=(5, 3))  # create a proportional-hazards plot.
plt.plot(h0_w, marker="o", label="x=0")  # baseline group.
plt.plot(h1_w, marker="o", label="x=1")  # higher-risk group.
plt.xlabel("time interval"); plt.ylabel("hazard")  # label axes.
plt.title("4: Cox scales the baseline hazard"); plt.legend(); plt.show()  # render.

▶ What you'll see: the two hazard curves have the same shape, but the x=1 curve is vertically multiplied.

*Why it's done this way:* Cox separates an unspecified time effect from covariate effects. That lets us estimate relative risks without committing to a parametric baseline hazard, as long as feature effects are multiplicative and stable over time.

### 5. Partial likelihood from risk sets

Cox learns $\beta$ without estimating $h_0(t)$ by comparing the subject who failed at each event time against everyone still at risk then. For an event at time $t_i$, the contribution is $$\frac{e^{\beta x_i}}{\sum_{j:t_j\ge t_i}e^{\beta x_j}}.$$ The baseline hazard cancels because every member of the same risk set shares the same $h_0(t_i)$.

In [ ]:
beta_grid_w = np.linspace(-1.0, 1.5, 101)  # candidate coefficients.
neglog_w = []  # negative partial log-likelihood values.
for b_w in beta_grid_w:
    total_w = 0.0  # log partial likelihood accumulator.
    for i_w in np.where(event_w == 1)[0]:
        risk_w = time_w >= time_w[i_w]  # people still at risk at this event time.
        scores_w = np.exp(b_w * x_w[risk_w])  # relative risks in the risk set.
        total_w += b_w * x_w[i_w] - np.log(np.sum(scores_w))  # event log probability.
    neglog_w.append(-total_w)  # minimize negative log likelihood.
neglog_w = np.array(neglog_w)  # array for argmin.
print("best beta on grid:", round(float(beta_grid_w[np.argmin(neglog_w)]), 3))  # inspect estimate.

▶ What you'll see: the best beta is negative because, in this tiny data, x=1 tends not to fail early.

In [ ]:
best_beta_w = float(beta_grid_w[np.argmin(neglog_w)])  # choose the grid optimum.
best_hr_w = float(np.exp(best_beta_w))  # convert coefficient to hazard ratio.
print("best hazard ratio:", round(best_hr_w, 3))  # less than 1 means x=1 appears lower risk.
assert best_hr_w < 1.0  # concrete check of the fitted direction.
plt.figure(figsize=(5, 3))  # create likelihood curve plot.
plt.plot(beta_grid_w, neglog_w, color="navy")  # plot objective.
plt.axvline(best_beta_w, color="crimson", linestyle="--", label=f"best β={best_beta_w:.2f}")  # mark optimum.
plt.xlabel("β"); plt.ylabel("negative partial log-likelihood"); plt.legend()  # label.
plt.title("5: Cox objective from risk sets"); plt.show()  # display.

▶ What you'll see: the curve bottoms where event ordering is best explained by the feature.

*Why it's done this way:* at each event time, the model only needs to rank the failing subject against the current risk set. That is why censored subjects still matter before their censoring time and why the unknown baseline hazard cancels out of the ratio.

### 6. Empirical score, cost, and model choice

The lesson's ERM framing says the training quantity is an average loss, but selection should include the method's cost or stabilizing constraint. For the verified toy arithmetic, the per-example losses are 0.257, 0.096, and 0.539, the cost is 0.090, a flexible alternative scores 0.439, and a stabilizing knob reduces the baseline decision score by 20%.

In [ ]:
losses_w = np.array([0.257, 0.096, 0.539])  # verified per-example losses from the lesson block.
R_S_w = float(np.mean(losses_w))  # empirical risk = average loss.
cost_w = 0.090  # complexity, regularization, or operational cost.
score_w = R_S_w + cost_w  # selection score.
print("empirical risk:", round(R_S_w, 3))  # 0.297.
print("score with cost:", round(score_w, 3))  # 0.387.
assert round(R_S_w, 3) == 0.297 and round(score_w, 3) == 0.387  # verified lesson numbers.

▶ What you'll see: the raw average is not the final selection score once cost is included.

In [ ]:
alt_w = 0.439  # tempting flexible alternative.
gap_w = alt_w - score_w  # absolute advantage of the lower-cost model.
rel_gap_w = gap_w / alt_w  # relative gap.
stable_w = 0.80 * score_w  # stabilizing knob reduces the score by 20%.
choices_w = np.array([score_w, alt_w, stable_w])  # final candidate scores.
print("gap:", round(gap_w, 3), "relative gap:", round(rel_gap_w, 3))  # inspect evidence size.
print("stable score:", round(stable_w, 3), "winner index:", int(np.argmin(choices_w)))  # inspect winner.
assert round(gap_w, 3) == 0.052 and round(rel_gap_w, 3) == 0.118 and round(stable_w, 3) == 0.310  # verified numbers.

▶ What you'll see: the stabilized model has the lowest final score, but the gap quantifies how strong the preference is.

In [ ]:
plt.figure(figsize=(5, 3))  # create a model-selection chart.
plt.bar(["baseline+cost", "flexible alt", "stabilized"], choices_w, color=["steelblue", "gray", "seagreen"])  # compare candidates.
plt.ylabel("decision score (lower is better)"); plt.xticks(rotation=10)  # label.
plt.title("6: choose by full score, not raw fit"); plt.show()  # display.

▶ What you'll see: the green stabilized bar is lowest; that is the candidate to carry forward in this toy case.

*Why it's done this way:* ERM supplies the average-loss frame, but survival models can overfit event orderings when samples are small. Adding cost, checking gaps, and preferring stable scores keeps the decision tied to future performance rather than the prettiest training fragment.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, exponentials, sorting, masks, products, and numerical checks.
import matplotlib.pyplot as plt # load Matplotlib for survival curves, risk-set plots, and diagnostic charts.
np.random.seed(0) # make all random examples reproducible across notebook runs.

def km_curve(times, events): # compute a Kaplan-Meier step curve from observed times and event indicators.
    event_times = np.unique(times[events == 1]) # keep only times where an event occurred.
    surv = [] # store survival after each event time.
    s = 1.0 # survival starts at one.
    for t in event_times: # update at each event time.
        n = np.sum(times >= t) # count subjects still at risk just before t.
        d = np.sum((times == t) & (events == 1)) # count events at t.
        s *= (1 - d / n) # multiply conditional survival probabilities.
        surv.append(s) # store the updated survival.
    return event_times, np.array(surv) # return event times and survival values.

def cox_neg_loglik(beta, times, events, x): # compute one-feature Cox negative partial log-likelihood.
    total = 0.0 # log partial likelihood accumulator.
    for i in np.where(events == 1)[0]: # only event rows contribute numerator terms.
        risk = times >= times[i] # risk set: subjects observed at least to this event time.
        total += beta * x[i] - np.log(np.sum(np.exp(beta * x[risk]))) # log event probability.
    return -float(total) # minimize negative log likelihood.

def brier_at(t0, times, events, pred_surv): # compute a simple uncensored-at-t0 Brier score demo.
    usable = (times > t0) | ((times <= t0) & (events == 1)) # exclude censored before t0 because status at t0 is unknown.
    y = (times[usable] > t0).astype(float) # 1 if event-free past t0, else 0.
    return float(np.mean((y - pred_surv[usable]) ** 2)), usable # return score and used rows.

## 🟢 Basics (warm-up)

### Basic 1 — Build censored survival rows

**Goal.** Store times, event indicators, and one feature, because survival analysis must distinguish event times from censoring times. We build it in 2 steps.

In [ ]:
time_b1 = np.array([2., 3., 4., 6., 7., 9.]) # observed event or censoring times.
event_b1 = np.array([1, 0, 1, 1, 0, 1]) # 1 means event happened; 0 means right-censored.
x_b1 = np.array([0., 1., 0., 1., 1., 0.]) # one binary covariate for risk modeling.
print("times:", time_b1) # inspect observed durations.
print("events:", event_b1) # inspect event/censor labels.

▶ What you'll see: every row has a time, but only rows with event=1 are actual failures.

In [ ]:
censored_b1 = np.where(event_b1 == 0)[0] # find censored rows.
print("censored row indices:", censored_b1) # inspect rows with unknown future event times.
assert censored_b1.tolist() == [1, 4] # verify the two censored rows.
plt.figure(figsize=(5, 2.8)) # create a compact timeline.
plt.scatter(time_b1, np.arange(len(time_b1)), c=np.where(event_b1 == 1, "crimson", "gray"), s=80) # draw event/censor dots.
plt.title("Basic 1: observed times"); plt.xlabel("time"); plt.ylabel("row"); plt.show() # display.

▶ What you'll see: gray censored dots are last observation times, not event times.

👀 Takeaway: a survival row needs both a time and an event indicator.

### Basic 2 — Count who is at risk at each time

**Goal.** Build risk sets by time, because survival likelihoods compare events only against subjects still under observation. We build it in 2 steps.

In [ ]:
time_b2 = np.array([2., 3., 4., 6., 7., 9.]) # observed times.
event_b2 = np.array([1, 0, 1, 1, 0, 1]) # event indicators.
query_times_b2 = np.array([2., 4., 6., 9.]) # event times to inspect.
risk_counts_b2 = np.array([np.sum(time_b2 >= t_b2) for t_b2 in query_times_b2]) # count at-risk rows.
print("risk counts:", risk_counts_b2) # inspect denominators.

▶ What you'll see: the risk set shrinks as time advances.

In [ ]:
assert risk_counts_b2.tolist() == [6, 4, 3, 1] # verify the risk-set counts.
plt.figure(figsize=(4, 3)) # create a risk-count chart.
plt.bar([str(t_b2) for t_b2 in query_times_b2], risk_counts_b2, color="teal") # plot at-risk counts.
plt.title("Basic 2: risk set sizes"); plt.xlabel("event time"); plt.ylabel("at risk") # label.
plt.show() # display.

▶ What you'll see: later event times have fewer comparable subjects.

👀 Takeaway: censoring removes a subject from future risk sets after their observed time.

### Basic 3 — Compute one Kaplan–Meier step

**Goal.** Calculate the first KM update, because survival drops by the event fraction among those at risk. We build it in 2 steps.

In [ ]:
time_b3 = np.array([2., 3., 4., 6., 7., 9.]) # observed times.
event_b3 = np.array([1, 0, 1, 1, 0, 1]) # event indicators.
t_b3 = 2.0 # first event time.
n_b3 = np.sum(time_b3 >= t_b3) # subjects at risk just before t=2.
d_b3 = np.sum((time_b3 == t_b3) & (event_b3 == 1)) # events at t=2.
print("n:", int(n_b3), "d:", int(d_b3)) # inspect KM ingredients.

▶ What you'll see: all 6 subjects are at risk and 1 event occurs.

In [ ]:
S_after_b3 = 1.0 * (1 - d_b3 / n_b3) # first KM survival update.
print("survival after first event:", round(float(S_after_b3), 3)) # inspect survival.
assert round(float(S_after_b3), 3) == 0.833 # verify 5/6.
plt.figure(figsize=(4, 3)) # create a one-step plot.
plt.step([0, t_b3], [1.0, S_after_b3], where="post", color="purple") # show first drop.
plt.ylim(0, 1.05); plt.title("Basic 3: first KM drop"); plt.show() # display.

▶ What you'll see: survival drops from 1.000 to 0.833 at the first event.

👀 Takeaway: a KM drop is `events / at-risk`, not `events / all rows forever`.

### Basic 4 — Estimate a full KM curve

**Goal.** Compute all KM steps, because the survival curve is a product of conditional survival factors. We build it in 2 steps.

In [ ]:
time_b4 = np.array([2., 3., 4., 6., 7., 9.]) # observed times.
event_b4 = np.array([1, 0, 1, 1, 0, 1]) # event indicators.
km_t_b4, km_s_b4 = km_curve(time_b4, event_b4) # compute KM event times and survival values.
print("KM times:", km_t_b4) # inspect event times.
print("KM survival:", np.round(km_s_b4, 3)) # inspect survival steps.

▶ What you'll see: survival values are updated only at event times.

In [ ]:
assert round(float(km_s_b4[1]), 3) == 0.625 # verify the second step.
plt.figure(figsize=(5, 3)) # create a KM curve plot.
plt.step(np.r_[0, km_t_b4], np.r_[1.0, km_s_b4], where="post", color="navy") # draw survival.
plt.ylim(0, 1.05); plt.xlabel("time"); plt.ylabel("S(t)") # label.
plt.title("Basic 4: Kaplan-Meier curve"); plt.show() # display.

▶ What you'll see: flat segments persist through censored times, with drops at event times.

👀 Takeaway: KM is a censoring-aware nonparametric survival estimate.

### Basic 5 — Convert a Cox coefficient to a hazard ratio

**Goal.** Interpret β through exp(β), because Cox covariates multiply the baseline hazard. We build it in 2 steps.

In [ ]:
beta_b5 = 0.7 # one-feature Cox coefficient.
hr_b5 = float(np.exp(beta_b5)) # hazard ratio for a one-unit feature increase.
print("hazard ratio:", round(hr_b5, 3)) # inspect multiplicative risk.
assert round(hr_b5, 3) == 2.014 # verify exp(0.7).

▶ What you'll see: a coefficient of 0.7 means about 2.014× hazard.

In [ ]:
base_b5 = np.array([0.04, 0.06, 0.08]) # baseline hazard over three intervals.
scaled_b5 = base_b5 * hr_b5 # Cox hazard for x=1.
print("scaled hazards:", np.round(scaled_b5, 3)) # inspect multiplied hazards.
plt.figure(figsize=(4, 3)) # create a small comparison.
plt.plot(base_b5, marker="o", label="x=0"); plt.plot(scaled_b5, marker="o", label="x=1") # plot.
plt.title("Basic 5: hazard ratio scaling"); plt.legend(); plt.show() # display.

▶ What you'll see: the x=1 hazard curve is the baseline curve multiplied by the same factor.

👀 Takeaway: Cox β is easiest to explain as a multiplicative hazard ratio.

### Basic 6 — Compute linear risk scores

**Goal.** Form βx and exp(βx), because Cox compares relative risks inside each risk set. We build it in 2 steps.

In [ ]:
x_b6 = np.array([0., 1., 2.]) # three feature values.
beta_b6 = 0.4 # coefficient.
linear_b6 = beta_b6 * x_b6 # linear risk scores.
relrisk_b6 = np.exp(linear_b6) # relative risks.
print("linear scores:", linear_b6) # inspect βx.
print("relative risks:", np.round(relrisk_b6, 3)) # inspect exp(βx).

▶ What you'll see: each one-unit increase multiplies risk by exp(0.4).

In [ ]:
assert round(float(relrisk_b6[2]), 3) == 2.226 # verify exp(0.8).
plt.figure(figsize=(4, 3)) # create a relative-risk bar chart.
plt.bar(["x=0", "x=1", "x=2"], relrisk_b6, color="orange") # plot relative risks.
plt.title("Basic 6: relative risk exp(βx)"); plt.ylabel("relative risk") # label.
plt.show() # display.

▶ What you'll see: relative risk grows exponentially with the linear score.

👀 Takeaway: Cox is linear in log-risk and multiplicative in hazard.

### Basic 7 — Build one Cox risk set denominator

**Goal.** Compute the denominator for one event probability, because partial likelihood normalizes over everyone still at risk. We build it in 2 steps.

In [ ]:
time_b7 = np.array([2., 3., 4., 6., 7., 9.]) # observed times.
x_b7 = np.array([0., 1., 0., 1., 1., 0.]) # one feature.
beta_b7 = 0.5 # chosen coefficient.
event_index_b7 = 2 # row with event at time 4.
risk_b7 = time_b7 >= time_b7[event_index_b7] # rows at risk at t=4.
print("risk set rows:", np.where(risk_b7)[0]) # inspect denominator members.

▶ What you'll see: only rows observed to time 4 or later are in the risk set.

In [ ]:
den_b7 = float(np.sum(np.exp(beta_b7 * x_b7[risk_b7]))) # denominator sum of relative risks.
num_b7 = float(np.exp(beta_b7 * x_b7[event_index_b7])) # numerator for the failing row.
prob_b7 = num_b7 / den_b7 # partial-likelihood event probability.
print("denominator:", round(den_b7, 3), "event probability:", round(prob_b7, 3)) # inspect.
assert round(den_b7, 3) == 5.297 # 2*exp(.5)+2*exp(0).

▶ What you'll see: the event probability is the failing subject's relative risk divided by risk-set total.

👀 Takeaway: Cox training is built from event-versus-risk-set comparisons.

### Basic 8 — Evaluate a negative partial log-likelihood

**Goal.** Score one β value, because fitting Cox means minimizing negative partial log-likelihood. We build it in 2 steps.

In [ ]:
time_b8 = np.array([2., 3., 4., 6., 7., 9.]) # observed times.
event_b8 = np.array([1, 0, 1, 1, 0, 1]) # event indicators.
x_b8 = np.array([0., 1., 0., 1., 1., 0.]) # feature.
beta_b8 = 0.0 # neutral coefficient.
nll_b8 = cox_neg_loglik(beta_b8, time_b8, event_b8, x_b8) # score β=0.
print("negative log likelihood:", round(nll_b8, 3)) # inspect objective value.

▶ What you'll see: β=0 treats every at-risk subject as equally likely to fail.

In [ ]:
assert round(nll_b8, 3) == 4.277 # log(6)+log(4)+log(3)+log(1).
plt.figure(figsize=(4, 3)) # create a simple objective marker.
plt.bar(["β=0"], [nll_b8], color="steelblue") # plot the objective value.
plt.title("Basic 8: Cox objective value"); plt.ylabel("negative log likelihood") # label.
plt.show() # display.

▶ What you'll see: the bar is the loss that other β values try to beat.

👀 Takeaway: lower Cox negative partial log-likelihood means better event ordering.

### Basic 9 — Compute empirical risk plus cost

**Goal.** Reproduce the lesson's average-loss score, because model selection should use the full decision score rather than raw fit only. We build it in 2 steps.

In [ ]:
losses_b9 = np.array([0.257, 0.096, 0.539]) # verified toy per-example losses.
R_S_b9 = float(np.mean(losses_b9)) # empirical risk.
cost_b9 = 0.090 # method cost or complexity charge.
score_b9 = R_S_b9 + cost_b9 # full selection score.
print("R_S:", round(R_S_b9, 3), "score:", round(score_b9, 3)) # inspect.

▶ What you'll see: the raw average is 0.297 and the cost-aware score is 0.387.

In [ ]:
assert round(R_S_b9, 3) == 0.297 and round(score_b9, 3) == 0.387 # verify lesson arithmetic.
plt.figure(figsize=(4, 3)) # create a score-breakdown plot.
plt.bar(["risk", "cost", "total"], [R_S_b9, cost_b9, score_b9], color=["teal", "gray", "purple"]) # plot components.
plt.title("Basic 9: raw risk plus cost"); plt.show() # display.

▶ What you'll see: the total score includes a visible cost component.

👀 Takeaway: the selection criterion is the full score implied by the method.

### Basic 10 — Pick the lowest decision score

**Goal.** Compare baseline, flexible, and stabilized scores, because the final choice is a score comparison. We build it in 2 steps.

In [ ]:
scores_b10 = np.array([0.387, 0.439, 0.310]) # baseline+cost, flexible alternative, stabilized candidate.
labels_b10 = np.array(["baseline", "flexible", "stabilized"]) # candidate names.
best_b10 = int(np.argmin(scores_b10)) # lower score wins.
print("scores:", scores_b10) # inspect candidates.
print("winner:", labels_b10[best_b10]) # inspect selected model.

▶ What you'll see: the stabilized candidate has the lowest score.

In [ ]:
assert labels_b10[best_b10] == "stabilized" # verify the lesson decision.
plt.figure(figsize=(5, 3)) # create a model comparison chart.
plt.bar(labels_b10, scores_b10, color=["steelblue", "gray", "seagreen"]) # plot scores.
plt.ylabel("decision score"); plt.title("Basic 10: lower score wins") # label.
plt.show() # display.

▶ What you'll see: the lowest bar marks the model to carry forward.

👀 Takeaway: survival modeling decisions should compare full, same-scale scores.

## 🟡 Easy

### Easy 1 — Plot KM curves by group

**Goal.** Compare nonparametric survival by feature group, because before fitting Cox we should inspect whether groups separate over time. We build it in 3 steps.

In [ ]:
time_e1 = np.array([2., 3., 4., 6., 7., 9., 5., 8.]) # observed times.
event_e1 = np.array([1, 0, 1, 1, 0, 1, 1, 0]) # event indicators.
group_e1 = np.array([0, 1, 0, 1, 1, 0, 1, 0]) # binary group.
print("group counts:", np.bincount(group_e1)) # inspect sample sizes.

▶ What you'll see: both groups have four rows.

In [ ]:
curves_e1 = [] # store group KM curves.
for g_e1 in [0, 1]: # compute a curve per group.
    m_e1 = group_e1 == g_e1 # select group rows.
    t_e1, s_e1 = km_curve(time_e1[m_e1], event_e1[m_e1]) # compute KM within group.
    curves_e1.append((t_e1, s_e1)) # store for plotting.
    print("group", g_e1, "KM:", np.round(s_e1, 3)) # inspect survival steps.

In [ ]:
plt.figure(figsize=(5, 3)) # create group survival plot.
for g_e1, (t_e1, s_e1) in enumerate(curves_e1): # draw each group.
    plt.step(np.r_[0, t_e1], np.r_[1.0, s_e1], where="post", label=f"group {g_e1}") # plot KM.
plt.ylim(0, 1.05); plt.xlabel("time"); plt.ylabel("S(t)"); plt.legend() # label.
plt.title("Easy 1: KM curves by group"); plt.show() # display.

▶ What you'll see: the group with earlier events drops sooner, suggesting different risk.

👀 Takeaway: grouped KM curves are a quick visual check before proportional-hazards modeling.

### Easy 2 — Fit a one-feature Cox coefficient by grid search

**Goal.** Minimize the Cox negative partial log-likelihood over a grid, because the coefficient is chosen by event ordering within risk sets. We build it in 3 steps.

In [ ]:
time_e2 = np.array([2., 3., 4., 6., 7., 9.]) # observed times.
event_e2 = np.array([1, 0, 1, 1, 0, 1]) # event indicators.
x_e2 = np.array([0., 1., 0., 1., 1., 0.]) # one feature.
grid_e2 = np.linspace(-1.5, 1.5, 121) # coefficient candidates.
print("grid size:", len(grid_e2)) # inspect search resolution.

▶ What you'll see: the grid tests 121 possible β values.

In [ ]:
nll_e2 = np.array([cox_neg_loglik(b_e2, time_e2, event_e2, x_e2) for b_e2 in grid_e2]) # score all β values.
best_i_e2 = int(np.argmin(nll_e2)) # locate best coefficient.
beta_hat_e2 = float(grid_e2[best_i_e2]) # read best β.
print("beta_hat:", round(beta_hat_e2, 3), "hazard ratio:", round(float(np.exp(beta_hat_e2)), 3)) # inspect fit.
assert beta_hat_e2 < 0 # verify the direction on this toy data.

In [ ]:
plt.figure(figsize=(5, 3)) # create objective curve.
plt.plot(grid_e2, nll_e2, color="navy") # draw negative log-likelihood.
plt.axvline(beta_hat_e2, color="crimson", linestyle="--") # mark best β.
plt.xlabel("β"); plt.ylabel("negative partial log-likelihood") # label.
plt.title("Easy 2: one-feature Cox grid search"); plt.show() # display.

▶ What you'll see: the minimum lies at a negative β, meaning x=1 appears protective in this tiny sample.

👀 Takeaway: Cox estimation fits relative event ordering, not a direct regression to observed times.

### Easy 3 — Convert Cox risk scores into survival curves

**Goal.** Combine a baseline cumulative hazard with Cox relative risks, because survival for x is $S(t|x)=\exp[-H_0(t)e^{\beta x}]$. We build it in 3 steps.

In [ ]:
H0_e3 = np.array([0.05, 0.12, 0.25, 0.40]) # baseline cumulative hazard values.
t_e3 = np.array([1., 2., 4., 6.]) # time grid.
beta_e3 = 0.7 # coefficient.
xvals_e3 = np.array([0., 1.]) # two covariate values.
print("baseline survival:", np.round(np.exp(-H0_e3), 3)) # inspect S0(t).

▶ What you'll see: baseline survival declines as cumulative hazard grows.

In [ ]:
surv_e3 = np.array([np.exp(-H0_e3 * np.exp(beta_e3 * x_e3)) for x_e3 in xvals_e3]) # Cox survival curves.
print("S(t|x=0):", np.round(surv_e3[0], 3)) # inspect low-risk curve.
print("S(t|x=1):", np.round(surv_e3[1], 3)) # inspect high-risk curve.
assert surv_e3[1, -1] < surv_e3[0, -1] # higher hazard means lower survival.

In [ ]:
plt.figure(figsize=(5, 3)) # create survival comparison.
plt.step(t_e3, surv_e3[0], where="post", label="x=0") # plot baseline covariate.
plt.step(t_e3, surv_e3[1], where="post", label="x=1") # plot higher-risk covariate.
plt.ylim(0, 1.05); plt.xlabel("time"); plt.ylabel("S(t|x)"); plt.legend() # label.
plt.title("Easy 3: Cox survival curves"); plt.show() # display.

▶ What you'll see: x=1 has lower survival at every time because its hazard is multiplied upward.

👀 Takeaway: Cox converts relative risk into a full survival curve once a baseline hazard is supplied.

### Easy 4 — Score a survival prediction with censoring-aware Brier loss

**Goal.** Compute a simple Brier score at one time point while excluding rows censored before that time, because their true status is unknown. We build it in 3 steps.

In [ ]:
time_e4 = np.array([2., 3., 4., 6., 7., 9.]) # observed times.
event_e4 = np.array([1, 0, 1, 1, 0, 1]) # event indicators.
pred_surv_e4 = np.array([0.30, 0.70, 0.45, 0.55, 0.80, 0.60]) # predicted probability of surviving past t0.
t0_e4 = 5.0 # evaluation horizon.
print("evaluation time:", t0_e4) # inspect horizon.

▶ What you'll see: predictions are probabilities of being event-free beyond time 5.

In [ ]:
brier_e4, usable_e4 = brier_at(t0_e4, time_e4, event_e4, pred_surv_e4) # compute usable rows and score.
print("usable rows:", np.where(usable_e4)[0]) # rows censored before t0 are excluded.
print("Brier score:", round(brier_e4, 3)) # inspect mean squared probability error.
assert round(brier_e4, 3) == 0.139 # verify the worked score.

In [ ]:
truth_e4 = (time_e4[usable_e4] > t0_e4).astype(float) # true survival status at t0 for usable rows.
plt.figure(figsize=(5, 3)) # create prediction-vs-truth plot.
plt.scatter(pred_surv_e4[usable_e4], truth_e4, color="purple", s=80) # plot probabilities against outcomes.
plt.xlabel("predicted S(5)"); plt.ylabel("actual survived past 5") # label.
plt.title("Easy 4: Brier inputs at t=5"); plt.show() # display.

▶ What you'll see: rows censored before 5 are absent because their time-5 status cannot be verified.

👀 Takeaway: survival validation must respect what censoring makes unknowable.

### Easy 5 — Compare raw, costed, and stabilized selection scores

**Goal.** Recreate the lesson's model-selection arithmetic, because the lower training fragment is not enough unless it survives cost and stability checks. We build it in 3 steps.

In [ ]:
losses_e5 = np.array([0.257, 0.096, 0.539]) # per-example losses.
raw_e5 = float(np.mean(losses_e5)) # empirical risk.
cost_e5 = 0.090 # cost term.
base_score_e5 = raw_e5 + cost_e5 # full baseline score.
print("raw:", round(raw_e5, 3), "base score:", round(base_score_e5, 3)) # inspect.

▶ What you'll see: cost raises the selection score from 0.297 to 0.387.

In [ ]:
flex_e5 = 0.439 # flexible alternative score.
stable_e5 = 0.80 * base_score_e5 # stabilized score.
all_scores_e5 = np.array([base_score_e5, flex_e5, stable_e5]) # candidate scores.
print("candidate scores:", np.round(all_scores_e5, 3)) # inspect all candidates.
assert np.allclose(np.round(all_scores_e5, 3), np.array([0.387, 0.439, 0.310])) # verify lesson values.

In [ ]:
labels_e5 = ["baseline", "flexible", "stabilized"] # candidate names.
plt.figure(figsize=(5, 3)) # create comparison chart.
plt.bar(labels_e5, all_scores_e5, color=["steelblue", "gray", "seagreen"]) # plot scores.
plt.title("Easy 5: full-score model choice"); plt.ylabel("lower is better") # label.
plt.show() # display.

▶ What you'll see: stabilization produces the lowest score in the verified toy calculation.

👀 Takeaway: model choice uses the same-scale final score, not whichever component looks best alone.

## 🔴 Advanced

### Advanced 1 — Detect non-proportional hazards visually

**Goal.** Simulate two groups whose hazards cross, because Cox proportional hazards assumes one group's hazard is a constant multiplier of the other's. We build it in 3 steps.

In [ ]:
t_a1 = np.arange(1, 7) # six time intervals.
h0_a1 = np.array([0.05, 0.06, 0.08, 0.12, 0.15, 0.18]) # group 0 hazard rises slowly.
h1_a1 = np.array([0.16, 0.14, 0.11, 0.09, 0.07, 0.06]) # group 1 hazard starts high then falls.
ratio_a1 = h1_a1 / h0_a1 # time-varying hazard ratio.
print("hazard ratios:", np.round(ratio_a1, 2)) # inspect PH violation.

▶ What you'll see: the hazard ratio changes a lot over time instead of staying constant.

In [ ]:
S0_a1 = np.exp(-np.cumsum(h0_a1)) # survival for group 0.
S1_a1 = np.exp(-np.cumsum(h1_a1)) # survival for group 1.
print("final survival group0:", round(float(S0_a1[-1]), 3), "group1:", round(float(S1_a1[-1]), 3)) # inspect.
assert ratio_a1[0] > 3 and ratio_a1[-1] < 0.4 # verify a crossing-risk pattern.

In [ ]:
fig_a1, ax_a1 = plt.subplots(1, 2, figsize=(8, 3)) # create side-by-side diagnostics.
ax_a1[0].plot(t_a1, h0_a1, marker="o", label="group 0"); ax_a1[0].plot(t_a1, h1_a1, marker="o", label="group 1") # hazards.
ax_a1[0].set_title("hazards"); ax_a1[0].legend() # label first plot.
ax_a1[1].plot(t_a1, ratio_a1, marker="o", color="crimson"); ax_a1[1].axhline(np.exp(0.0), color="gray", linestyle="--") # ratio.
ax_a1[1].set_title("time-varying hazard ratio"); plt.suptitle("Advanced 1: non-PH pattern"); plt.show() # display.

▶ What you'll see: hazards cross and the ratio is not flat, warning that a single Cox β is too simple.

👀 Takeaway: proportional hazards is a modeling assumption that should be checked, not assumed.

### Advanced 2 — Add L2 regularization to Cox selection

**Goal.** Sweep β with and without an L2 penalty, because small survival datasets can prefer unstable large coefficients. We build it in 3 steps.

In [ ]:
time_a2 = np.array([2., 3., 4., 6., 7., 9.]) # observed times.
event_a2 = np.array([1, 0, 1, 1, 0, 1]) # event indicators.
x_a2 = np.array([0., 1., 0., 1., 1., 0.]) # feature.
grid_a2 = np.linspace(-2.0, 2.0, 161) # coefficient grid.
lam_a2 = 0.2 # L2 penalty strength.
print("lambda:", lam_a2) # inspect regularization strength.

▶ What you'll see: the sweep will compare unpenalized and penalized objectives.

In [ ]:
plain_a2 = np.array([cox_neg_loglik(b_a2, time_a2, event_a2, x_a2) for b_a2 in grid_a2]) # unpenalized objective.
pen_a2 = plain_a2 + lam_a2 * grid_a2 ** 2 # add L2 coefficient cost.
b_plain_a2 = float(grid_a2[np.argmin(plain_a2)]) # unpenalized optimum.
b_pen_a2 = float(grid_a2[np.argmin(pen_a2)]) # penalized optimum.
print("plain beta:", round(b_plain_a2, 3), "penalized beta:", round(b_pen_a2, 3)) # inspect shrinkage.
assert abs(b_pen_a2) < abs(b_plain_a2) # verify regularization shrinks magnitude.

In [ ]:
plt.figure(figsize=(5, 3)) # create objective comparison.
plt.plot(grid_a2, plain_a2, label="partial NLL", color="gray") # unpenalized.
plt.plot(grid_a2, pen_a2, label="NLL + λβ²", color="teal") # penalized.
plt.axvline(b_pen_a2, color="crimson", linestyle="--", label="penalized best") # mark.
plt.xlabel("β"); plt.ylabel("objective"); plt.legend() # label.
plt.title("Advanced 2: regularized Cox objective"); plt.show() # display.

▶ What you'll see: the penalty lifts large |β| values and moves the optimum closer to zero.

👀 Takeaway: regularization trades some raw fit for stability when event data are limited.

### Advanced 3 — Bootstrap the Cox coefficient

**Goal.** Resample subjects and refit β, because a small validation gap can vanish under sampling noise. We build it in 3 steps.

In [ ]:
time_a3 = np.array([2., 3., 4., 6., 7., 9., 5., 8.]) # observed times.
event_a3 = np.array([1, 0, 1, 1, 0, 1, 1, 0]) # event indicators.
x_a3 = np.array([0., 1., 0., 1., 1., 0., 1., 0.]) # feature.
grid_a3 = np.linspace(-1.5, 1.5, 121) # fit grid.
rng_a3 = np.random.default_rng(3) # reproducible bootstrap.
print("subjects:", len(time_a3)) # inspect sample size.

▶ What you'll see: the bootstrap repeatedly samples eight subjects with replacement.

In [ ]:
boots_a3 = [] # store bootstrap β estimates.
for _a3 in range(80): # run a modest bootstrap.
    idx_a3 = rng_a3.integers(0, len(time_a3), len(time_a3)) # resample row indices.
    vals_a3 = np.array([cox_neg_loglik(b_a3, time_a3[idx_a3], event_a3[idx_a3], x_a3[idx_a3]) for b_a3 in grid_a3]) # score grid.
    boots_a3.append(float(grid_a3[np.argmin(vals_a3)])) # store best β.
boots_a3 = np.array(boots_a3) # convert to array.
print("bootstrap beta mean:", round(float(np.mean(boots_a3)), 3), "sd:", round(float(np.std(boots_a3)), 3)) # inspect uncertainty.

In [ ]:
ci_a3 = np.percentile(boots_a3, [5, 95]) # rough interval.
print("90% bootstrap interval:", np.round(ci_a3, 3)) # inspect stability range.
plt.figure(figsize=(5, 3)) # create histogram.
plt.hist(boots_a3, bins=15, color="slateblue", edgecolor="white") # plot β samples.
plt.axvline(ci_a3[0], color="crimson", linestyle="--"); plt.axvline(ci_a3[1], color="crimson", linestyle="--") # interval.
plt.title("Advanced 3: bootstrap β uncertainty"); plt.xlabel("β estimate"); plt.show() # display.

▶ What you'll see: β estimates spread across a range, showing how noisy small survival samples can be.

👀 Takeaway: apparent Cox effects and score gaps need uncertainty checks before being trusted.

### Advanced 4 — Validate two survival models at a fixed horizon

**Goal.** Compare two predicted survival models with a censoring-aware horizon score, because future performance should choose between alternatives. We build it in 3 steps.

In [ ]:
time_a4 = np.array([2., 3., 4., 6., 7., 9.]) # observed times.
event_a4 = np.array([1, 0, 1, 1, 0, 1]) # event indicators.
t0_a4 = 5.0 # validation horizon.
pred_simple_a4 = np.array([0.35, 0.65, 0.40, 0.55, 0.75, 0.70]) # model A survival probabilities.
pred_flex_a4 = np.array([0.10, 0.90, 0.90, 0.10, 0.10, 0.20]) # model B probabilities.
print("horizon:", t0_a4) # inspect validation time.

▶ What you'll see: both models predict probability of surviving past time 5.

In [ ]:
brier_simple_a4, usable_a4 = brier_at(t0_a4, time_a4, event_a4, pred_simple_a4) # score simple model.
brier_flex_a4, _ = brier_at(t0_a4, time_a4, event_a4, pred_flex_a4) # score flexible model.
gap_a4 = brier_flex_a4 - brier_simple_a4 # positive means simple is better.
print("simple:", round(brier_simple_a4, 3), "flexible:", round(brier_flex_a4, 3), "gap:", round(gap_a4, 3)) # inspect.
assert round(brier_simple_a4, 3) < round(brier_flex_a4, 3) # verify simple wins here.

In [ ]:
plt.figure(figsize=(5, 3)) # create validation comparison.
plt.bar(["simple", "flexible"], [brier_simple_a4, brier_flex_a4], color=["seagreen", "gray"]) # lower Brier is better.
plt.ylabel("Brier score at t=5"); plt.title("Advanced 4: validation horizon score") # label.
plt.show() # display.

▶ What you'll see: the lower bar is the model with better calibrated survival at the chosen horizon.

👀 Takeaway: survival models should be selected on validation criteria that match the decision horizon.

### Advanced 5 — Choose a stabilized Cox-style decision score

**Goal.** Combine empirical risk, cost, a flexible alternative, and a stabilization knob, because the lesson's final decision uses a full score rather than a raw training number. We build it in 3 steps.

In [ ]:
losses_a5 = np.array([0.257, 0.096, 0.539]) # verified toy losses.
raw_a5 = float(np.mean(losses_a5)) # empirical risk.
cost_a5 = 0.090 # cost term.
base_a5 = raw_a5 + cost_a5 # baseline full score.
flex_a5 = 0.439 # flexible alternative score.
print("raw:", round(raw_a5, 3), "base:", round(base_a5, 3), "flex:", flex_a5) # inspect.

▶ What you'll see: the raw training average is lower than every full score because it omits cost.

In [ ]:
stable_grid_a5 = np.array([1.00, 0.95, 0.90, 0.80]) # possible stabilization multipliers.
stable_scores_a5 = stable_grid_a5 * base_a5 # candidate stabilized scores.
all_scores_a5 = np.r_[base_a5, flex_a5, stable_scores_a5] # collect candidates.
print("stable scores:", np.round(stable_scores_a5, 3)) # inspect stabilized choices.
assert round(float(stable_scores_a5[-1]), 3) == 0.310 # verify 20% reduction.

In [ ]:
labels_a5 = ["base", "flex", "stable 1.00", "stable .95", "stable .90", "stable .80"] # labels.
best_a5 = int(np.argmin(all_scores_a5)) # choose lowest score.
print("best label:", labels_a5[best_a5], "best score:", round(float(all_scores_a5[best_a5]), 3)) # inspect winner.
plt.figure(figsize=(7, 3)) # create final decision plot.
plt.bar(labels_a5, all_scores_a5, color=["steelblue", "gray", "lightgreen", "mediumseagreen", "seagreen", "darkgreen"]) # plot candidates.
plt.xticks(rotation=25); plt.ylabel("decision score"); plt.title("Advanced 5: stabilized score selection") # label.
plt.show() # display.

▶ What you'll see: the strongest stabilization multiplier gives the lowest score in this verified toy setup.

👀 Takeaway: the final survival-model decision should include fit, cost, uncertainty, and stabilization on one consistent scale.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Survival models learn time-to-event risk while respecting censoring.

Cox proportional hazards models estimate relative time-to-event risk without specifying the baseline hazard. The real difficulty is censoring: some examples leave observation before the event occurs. Save a copy to Drive to edit.

In [ ]:
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_diabetes
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.datasets import make_regression
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.linear_model import Ridge
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)
np.random.seed(7)

def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...].

    All X are 2-D float feature matrices, y integer labels, so one classifier runs unchanged
    across every rung (the 'watch it scale' story). Rungs get harder: clean+separable -> real
    high-dimensional. D1 is hand-built and fully inspectable.
    """
    rungs = []

    # D1 — four hand-placed 2-D points, 2 classes, clearly separable.
    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    # D2 — clean, well-separated Gaussian blobs.
    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    # D3 — non-linear, overlapping two-moons with noise.
    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    # D4 — real: Wine, 13 features, 3 classes.
    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    # D5 — real, harder: Breast Cancer, 30 features, class imbalance.
    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs

def reg_ladder():
    """D1..D5 regression ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0], [1.0], [2.0], [3.0]])
    y1 = np.array([1.0, 3.0, 5.0, 7.0])
    rungs.append(("D1 hand line y=2x+1", x1, y1))

    rng = np.random.default_rng(1)
    x2 = np.linspace(-3, 3, 120).reshape(-1, 1)
    y2 = (2.0 * x2[:, 0] + 1.0) + rng.normal(0, 0.5, size=120)
    rungs.append(("D2 linear + noise", x2, y2))

    x3 = np.linspace(-3, 3, 160).reshape(-1, 1)
    y3 = np.sin(1.5 * x3[:, 0]) + rng.normal(0, 0.2, size=160)
    rungs.append(("D3 sine (non-linear)", x3, y3))

    dia = load_diabetes()
    rungs.append(("D4 Diabetes (real, 10-D)", dia.data, dia.target))

    x5, y5 = make_regression(n_samples=300, n_features=20, n_informative=8, noise=25.0, random_state=5)
    rungs.append(("D5 high-dim + noise (20-D)", x5, y5))

    return rungs


def lesson_score(losses, cost, alternative):
    raw = float(np.sum(losses) / len(losses))
    score = raw + cost
    gap = alternative - score
    relative_gap = gap / alternative
    return raw, score, gap, relative_gap


def preview_ladder(rungs, is_regression=False):
    rows = []
    for index, item in enumerate(rungs, start=1):
        name, X, y = item
        if is_regression:
            info = f"target range {np.min(y):.2f}..{np.max(y):.2f}"
        else:
            values, counts = np.unique(y, return_counts=True)
            pairs = [f"{int(v)}:{int(c)}" for v, c in zip(values, counts)]
            info = ", ".join(pairs)
        row = {"rung": f"D{index}", "name": name, "shape": X.shape, "info": info}
        rows.append(row)
        print(row)
    name, X, y = rungs[0]
    print("sample X:")
    print(np.round(X[:5], 3))
    print("sample y:")
    print(np.round(y[:5], 3))
    return rows


def two_dimensional_view(X):
    if X.shape[1] == 1:
        return np.c_[X[:, 0], np.zeros(X.shape[0])]
    if X.shape[1] == 2:
        return X
    view = PCA(n_components=2, random_state=0).fit_transform(StandardScaler().fit_transform(X))
    return view


def stream_batches(X, y, batch_size):
    rng = np.random.default_rng(11)
    order = rng.permutation(len(y))
    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        yield X[idx], y[idx]


def online_fit_predict(X, y, kind="sgd", epochs=8, batch_size=16):
    stratify = y if np.min(np.bincount(y.astype(int))) >= 2 else None
    x_train, x_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.4,
        random_state=3,
        stratify=stratify,
    )
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
    classes = np.unique(y)
    if kind == "pa":
        model = PassiveAggressiveClassifier(C=0.6, random_state=3, max_iter=1, tol=None)
    else:
        model = SGDClassifier(loss="log_loss", alpha=0.0005, random_state=3, learning_rate="optimal")
    first = True
    history = []
    batch_size = max(2, min(batch_size, len(y_train)))
    for epoch in range(epochs):
        for xb, yb in stream_batches(x_train, y_train, batch_size):
            if first:
                model.partial_fit(xb, yb, classes=classes)
                first = False
            else:
                model.partial_fit(xb, yb)
        preds = model.predict(x_test)
        history.append(float(accuracy_score(y_test, preds)))
    preds = model.predict(x_test)
    return model, scaler, x_train, x_test, y_train, y_test, preds, history


def logistic_accuracy(X, y, weighted=False):
    stratify = y if np.min(np.bincount(y.astype(int))) >= 2 else None
    x_train, x_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.4,
        random_state=4,
        stratify=stratify,
    )
    class_weight = None
    if weighted:
        class_weight = "balanced"
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight=class_weight, random_state=4),
    )
    model.fit(x_train, y_train)
    preds = model.predict(x_test)
    acc = float(accuracy_score(y_test, preds))
    return model, x_train, x_test, y_train, y_test, preds, acc


def expected_binary_cost(y_true, y_pred, false_negative_cost=5.0, false_positive_cost=1.0):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return float((false_negative_cost * fn + false_positive_cost * fp) / len(y_true))


def make_multi_targets(y):
    y = np.asarray(y, dtype=float)
    scale = np.std(y)
    if scale == 0:
        scale = 1.0
    centered = (y - np.mean(y)) / scale
    cuts = np.quantile(centered, [0.33, 0.66])
    ordinal = np.digitize(centered, cuts).astype(float)
    return np.c_[centered, ordinal]


def multioutput_fit_predict(X, y, alpha=1.0):
    targets = make_multi_targets(y)
    x_train, x_test, y_train, y_test = train_test_split(
        X,
        targets,
        test_size=0.4,
        random_state=5,
    )
    model = make_pipeline(
        StandardScaler(),
        MultiOutputRegressor(Ridge(alpha=alpha)),
    )
    model.fit(x_train, y_train)
    preds = model.predict(x_test)
    mse = float(mean_squared_error(y_test, preds))
    r2 = float(r2_score(y_test, preds, multioutput="variance_weighted"))
    return model, x_train, x_test, y_train, y_test, preds, mse, r2


def make_survival_from_classification(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=int)
    rng = np.random.default_rng(23 + X.shape[0] + X.shape[1])
    weights = np.linspace(0.4, 1.2, X.shape[1])
    linear = StandardScaler().fit_transform(X).dot(weights) / math.sqrt(X.shape[1])
    class_effect = (y == np.max(y)).astype(float) * 0.8
    risk = linear + class_effect
    event_time = np.exp(-0.45 * risk) + rng.gamma(shape=2.0, scale=0.25, size=len(y))
    censor_time = rng.gamma(shape=2.3, scale=0.5, size=len(y)) + 0.35
    observed_time = np.minimum(event_time, censor_time)
    event = (event_time <= censor_time).astype(int)
    if np.sum(event) < 3:
        event[:3] = 1
    return observed_time, event


def cox_fit(X, time, event, lr=0.03, steps=220, l2=0.02):
    X = np.asarray(X, dtype=float)
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    beta = np.zeros(X.shape[1])
    order = np.argsort(-time)
    X_desc = X[order]
    event_desc = event[order]
    for step in range(steps):
        scores = np.clip(X_desc.dot(beta), -30, 30)
        exp_scores = np.exp(scores)
        risk_sum = np.cumsum(exp_scores)
        weighted_sum = np.cumsum(exp_scores[:, None] * X_desc, axis=0)
        grad = np.zeros_like(beta)
        event_positions = np.where(event_desc == 1)[0]
        for pos in event_positions:
            grad += X_desc[pos] - weighted_sum[pos] / risk_sum[pos]
        grad = grad / max(1, len(event_positions))
        grad = grad - l2 * beta
        beta = beta + lr * grad
    return beta


def concordance_index(time, event, risk):
    total = 0
    good = 0.0
    for i in range(len(time)):
        for j in range(len(time)):
            if time[i] < time[j] and event[i] == 1:
                total += 1
                if risk[i] > risk[j]:
                    good += 1.0
                elif risk[i] == risk[j]:
                    good += 0.5
    if total == 0:
        return 0.5
    return float(good / total)


def survival_fit_score(X, y):
    time, event = make_survival_from_classification(X, y)
    stratify = y if np.min(np.bincount(y.astype(int))) >= 2 else None
    x_train, x_test, t_train, t_test, e_train, e_test = train_test_split(
        X,
        time,
        event,
        test_size=0.4,
        random_state=6,
        stratify=stratify,
    )
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
    beta = cox_fit(x_train, t_train, e_train)
    train_risk = x_train.dot(beta)
    test_risk = x_test.dot(beta)
    cindex = concordance_index(t_test, e_test, test_risk)
    return beta, scaler, x_train, x_test, t_train, t_test, e_train, e_test, train_risk, test_risk, cindex


def cross_validation_gap(X, y, k=5):
    counts = np.bincount(y.astype(int))
    min_count = int(np.min(counts))
    k = max(2, min(k, min_count))
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=8),
    )
    cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=8)
    result = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
    )
    train_loss = 1.0 - result["train_score"]
    val_loss = 1.0 - result["test_score"]
    gap = float(np.mean(val_loss - train_loss))
    return train_loss, val_loss, gap, cv

## The concept, built once (D1)

The lesson formula is

$$h(t\mid x)=h_0(t)e^{\beta^\top x}$$

Plug in the lesson losses 0.257, 0.096, and 0.539. The average is $R_S=0.892/3=0.297$, the cost is $0.090$, the score is $0.387$, and the alternative gap is $0.439-0.387=0.052$.

In [ ]:
def survival_analysis_cox_proportional_hazards_method():
    losses = np.array([0.257, 0.096, 0.539], dtype=float)
    cost = 0.090
    alternative = 0.439
    beta = np.array([0.6, -0.2])
    x = np.array([1.5, 0.5])
    baseline_hazard = 0.04
    risk_ratio = float(np.exp(np.dot(beta, x)))
    hazard = baseline_hazard * risk_ratio
    raw, score, gap, relative_gap = lesson_score(losses, cost, alternative)
    assert np.isclose(risk_ratio, np.exp(0.8))
    assert np.isclose(raw, 0.297333333333)
    assert np.isclose(score, 0.387333333333)
    assert np.isclose(gap, 0.051666666667)
    return {"risk_ratio": risk_ratio, "hazard": hazard, "raw": raw, "score": score, "gap": gap}

lesson_check = survival_analysis_cox_proportional_hazards_method()
print(lesson_check)

The method returns the arithmetic pieces and asserts the exact lesson numbers before any larger data appears.

In [ ]:
assert lesson_check['score'] > lesson_check['raw']
assert lesson_check['gap'] > 0
print('lesson arithmetic locked')

## The dataset ladder

Use the shared classification ladder so the same learner runs from a hand toy to real Breast Cancer features.

In [ ]:
rungs = clf_ladder()
ladder_preview = preview_ladder(rungs, is_regression=False)

## Run the same method across D1–D5

Only the data rung changes. The metric is the plan metric for this topic.

In [ ]:
results = []
artifacts = []
for rung_index, (name, X, y) in enumerate(rungs, start=1):
    beta, scaler, x_train, x_test, t_train, t_test, e_train, e_test, train_risk, test_risk, cindex = survival_fit_score(X, y)
    results.append({"rung": rung_index, "name": name, "cindex": cindex, "events": int(np.sum(e_test))})
    artifacts.append((name, X, y, t_test, e_test, test_risk, beta))
for row in results:
    print(f"D{row['rung']} {row['cindex']:.3f} concordance, events={row['events']} — {row['name']}")

## Results visualization

The first figure shows the model artifact on each rung. The second summarizes `cindex` from D1 through D5.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 3.8))
for index, artifact in enumerate(artifacts):
    name, X, y, t_test, e_test, test_risk, beta = artifact
    axes[index].scatter(t_test, test_risk, c=e_test, cmap="coolwarm", s=18, alpha=0.75)
    axes[index].set_title(f"D{index + 1}: risk vs time")
    axes[index].set_xlabel("observed time")
    axes[index].set_ylabel("Cox risk")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 3.5))
plt.plot([row["rung"] for row in results], [row["cindex"] for row in results], marker="o")
plt.xticks([1, 2, 3, 4, 5], ["D1", "D2", "D3", "D4", "D5"])
plt.ylim(0, 1.05)
plt.ylabel("concordance index")
plt.title("Cox concordance vs. ladder complexity")
plt.grid(True, alpha=0.3)
plt.show()

## Pitfall on the hardest rung

The lesson warning is to optimize the raw term and forget the cost. On D5, the raw metric alone can pick a different setting than the cost-aware score.

In [ ]:
name, X, y = rungs[-1]
time, event = make_survival_from_classification(X, y)
scaler = StandardScaler()
Xs = scaler.fit_transform(X)
plain_beta = cox_fit(Xs, time, event, lr=0.04, steps=260, l2=0.0)
regularized_beta = cox_fit(Xs, time, event, lr=0.04, steps=260, l2=0.08)
plain_c = concordance_index(time, event, Xs.dot(plain_beta))
regularized_c = concordance_index(time, event, Xs.dot(regularized_beta))
raw_only_winner = "plain" if plain_c >= regularized_c else "regularized"
plain_score = (1.0 - plain_c) + 0.090
regularized_score = (1.0 - regularized_c) + 0.090 * 0.5
cost_aware_winner = "plain" if plain_score <= regularized_score else "regularized"
print("D5 raw concordance", plain_c, regularized_c, "raw winner", raw_only_winner)
print("D5 cost-aware scores", plain_score, regularized_score, "cost-aware winner", cost_aware_winner)
print("lesson raw", 0.297, "cost", 0.090, "score", 0.387, "gap", 0.439 - 0.387)

## Evaluate it + Practice

- Compare the displayed metric with a no-skill baseline such as majority class, mean target, or random fold assignment.
- Sanity-check D1 by hand before trusting the D5 curve.
- Ablate the key idea: remove partial updates, remove PA margins, collapse outputs, ignore censoring, remove costs, or reuse the test set.
- Failure signals include unstable D5 metrics, a widening validation gap, or a cost-aware score that disagrees with the raw metric.

Practice 1: change the seed or batch/fold size and rerun the D1-to-D5 table.

Practice 2: turn off the topic-specific idea and measure the metric drop on D5.

Practice 3: add one extra diagnostic plot for the hardest rung.